In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import scanpy as sc
import anndata

from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import precision_recall_curve, average_precision_score
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
import pickle
from sklearn.multiclass import OneVsRestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

In [ ]:
sc.set_figure_params(figsize=(5,5), frameon=False)
sc.settings.verbosity = 3

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc'
donors = list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str))
tmp_order = ['Start', 'Sendai', 'Skin', 'Inflamed', 'Delayed', 'Fail1', 'Inter1',  'Inter2', 'Fail2','IPS']

In [ ]:
clusters_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/rna/csv/label_transfer/snmct_xgboost_v7.csv'
clusters = pd.read_csv(clusters_path, sep='\t', header=0, index_col=0)
clusters.shape

In [ ]:
clusters.head()

### load all of the feature sets

In [ ]:
train_size = 0.8
test_size = 0.2
model = OneVsRestClassifier(XGBClassifier(eval_metric='logloss', random_state=42))

In [ ]:
var_dims_path = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/mc/txt/dim_red/var_dims.txt'
var_dims = pd.read_csv(var_dims_path, header=None)[0].values
var_dims

In [ ]:
adatas = []
for v in var_dims:
    tmp_adata = sc.read_h5ad(f'{projdir}/h5ad/{v}/merged.processed.small.h5ad')
    tmp_adata.obs['cluster'] = clusters.reindex(tmp_adata.obs.index)['pred_coarse']
    adatas.append(tmp_adata)

In [ ]:
f1_scores = []
for tmp_adata in adatas:
    X = np.hstack([tmp_adata.obsm['X_pca'], tmp_adata.obs['time_num'].values.reshape(-1, 1)])
    y = tmp_adata.obs['cluster']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size, stratify=y, random_state=0)
    model.fit(X_train, y_train)

    tmp_pred = model.predict(X_test)

    tmp_f1 = f1_score(y_test, tmp_pred, labels=tmp_order, average=None)
    f1_scores.append(tmp_f1)

In [ ]:
merged_scores = pd.DataFrame(f1_scores, index=var_dims, columns=tmp_order)
merged_scores.shape

In [ ]:
merged_scores.head()

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv'
merged_scores.to_csv(f'{outdir}/figure1/mc_cluster_f1_scores_var_dims.csv', sep='\t')

### predict donor

In [ ]:
f1_scores = []
for tmp_adata in adatas:
    X = np.hstack([tmp_adata.obsm['X_pca'], tmp_adata.obs['time_num'].values.reshape(-1, 1)])
    y = tmp_adata.obs['donor']

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, train_size=train_size, stratify=y, random_state=0)
    model.fit(X_train, y_train)

    tmp_pred = model.predict(X_test)

    tmp_f1 = f1_score(y_test, tmp_pred, labels=donors, average=None)
    f1_scores.append(tmp_f1)

In [ ]:
merged_scores = pd.DataFrame(f1_scores, index=var_dims, columns=donors)
merged_scores.shape

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure2/'
merged_scores.to_csv(f'{outdir}/mc_donor_f1_scores.csv', sep='\t')

In [ ]:
!date